# Agglomerative hierarchical clustering
The proximity matrix by hand, the four linkages, and the shopping data: dendrogram, cut, 5 clusters.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from sklearn.cluster import AgglomerativeClustering

## The proximity matrix

In [2]:
P = np.array([[1, 1], [2, 2], [5, 4], [6, 4], [6, 6]])
D = cdist(P, P)                       # distance between every pair of points: a 5 x 5 matrix
pd.DataFrame(D.round(2), index=["P1", "P2", "P3", "P4", "P5"], columns=["P1", "P2", "P3", "P4", "P5"])

,P1,P2,P3,P4,P5
P1,0.00,1.41,5.00,5.83,7.07
P2,1.41,0.00,3.61,4.47,5.66
P3,5.00,3.61,0.00,1.00,2.24
P4,5.83,4.47,1.00,0.00,2.00
P5,7.07,5.66,2.24,2.00,0.00


In [3]:
# The merges, one row each: [cluster a, cluster b, distance, size]; ids 5, 6, ... are new clusters
for method in ["single", "complete", "average", "ward"]:
    print(method, "\n", linkage(P, method=method).round(2))

single 
 [[2.   3.   1.   2.  ]
 [0.   1.   1.41 2.  ]
 [4.   5.   2.   3.  ]
 [6.   7.   3.61 5.  ]]
complete 
 [[2.   3.   1.   2.  ]
 [0.   1.   1.41 2.  ]
 [4.   5.   2.24 3.  ]
 [6.   7.   7.07 5.  ]]
average 
 [[2.   3.   1.   2.  ]
 [0.   1.   1.41 2.  ]
 [4.   5.   2.12 3.  ]
 [6.   7.   5.27 5.  ]]
ward 
 [[2.   3.   1.   2.  ]
 [0.   1.   1.41 2.  ]
 [4.   5.   2.38 3.  ]
 [6.   7.   8.11 5.  ]]


## The shopping data

In [4]:
customers = pd.read_csv("data/shopping_data.csv")
print(customers.shape)
customers.head()

(200, 5)


,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [5]:
# Keep two columns: annual income and spending score
data = customers.iloc[:, 3:5].values

# Hierarchical clustering is distance-based, so put both columns on one scale first
from sklearn.preprocessing import StandardScaler
data_std = StandardScaler().fit_transform(data)

In [6]:
# The dendrogram: scipy computes it, Plotly draws it
Z = linkage(data_std, method="ward")
d = dendrogram(Z, no_plot=True)      # coordinates only, no plotting library needed
fig = go.Figure()
for xs, ys in zip(d["icoord"], d["dcoord"]):
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#4C78A8", width=1), showlegend=False))
fig.add_hline(y=7, line_dash="dash", line_color="red")
fig.update_layout(template="simple_white", title="Customer dendrogram", xaxis_showticklabels=False,
                  yaxis_title="merge distance")

In [7]:
# The last merge heights, largest first: the gaps between them are the uncut vertical lines
print(np.sort(Z[:, 2])[::-1][:6].round(2))

# For comparison: the same heights on the raw, unscaled features
print(np.sort(linkage(data, method="ward")[:, 2])[::-1][:6].round(1))

[15.62 15.19 10.14  9.45  4.35  4.31]
[405.7 394.9 262.6 245.7 113.9 112.8]


In [8]:
cluster = AgglomerativeClustering(n_clusters=5, metric="euclidean", linkage="ward")
labels_ = cluster.fit_predict(data_std)   # cluster on the scaled values; plot the original ones
print(labels_[:20])
print("customers per cluster:", np.bincount(labels_))

[4 3 4 3 4 3 4 3 4 3 4 3 4 3 4 3 4 3 4 3]
customers per cluster: [32 39 85 21 23]


In [9]:
fig = go.Figure()
for k in range(5):
    fig.add_trace(go.Scatter(x=data[labels_ == k, 0], y=data[labels_ == k, 1], mode="markers", name=f"cluster {k}"))
fig.update_layout(template="simple_white", xaxis_title="annual income (k$)", yaxis_title="spending score")

In [10]:
# Instead of n_clusters, a distance threshold: stop merging above this height
cut = AgglomerativeClustering(n_clusters=None, distance_threshold=7, linkage="ward").fit(data_std)
print("clusters below 7:", cut.n_clusters_)

clusters below 7: 5
